# Nom et Prenom : Mouhamed SAMB

#A. Implémentation de k-means séquentiel

## 1.Generation des données

In [ ]:
import numpy as np
from typing import Tuple, Iterator
import csv
import matplotlib.pyplot as plt

def generate_data(n_points_per_class: int = 100) -> None:
    """
    Génère un jeu de données avec deux classes et sauvegarde dans un fichier CSV.

    Args:
        n_points_per_class: Nombre de points par classe
    """
    # Génération des points pour la classe 1 autour de (5, 5)
    class1 = np.random.normal(loc=[5, 5], scale=1.0, size=(n_points_per_class, 2))

    # Génération des points pour la classe 2 autour de (10, 10)
    class2 = np.random.normal(loc=[10, 10], scale=1.0, size=(n_points_per_class, 2))

    # Combinaison des deux classes
    data = np.vstack((class1, class2))

    # Sauvegarde dans un fichier CSV
    with open('data.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['x', 'y'])  # En-têtes
        writer.writerows(data)

## 2.Lecture des données

In [ ]:
def read_data(filename: str) -> Iterator[np.ndarray]:
    """
    Lit les données depuis un fichier CSV de manière efficace en mémoire.

    Args:
        filename: Nom du fichier CSV

    Yields:
        Un point sous forme de numpy array
    """
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        next(reader)  # Skip header
        for row in reader:
            yield np.array([float(row[0]), float(row[1])])


## 3.Implémenter l’algorithme k-means séquentiel

In [ ]:
class KMeansSequential:
    def __init__(self, k: int):
        """
        Initialise l'algorithme k-means séquentiel.

        Args:
            k: Nombre de clusters
        """
        self.k = k
        self.centers = None
        self.counts = None

    def initialize_centers(self, first_k_points: list) -> None:
        """
        Initialise les centres avec les k premiers points.

        Args:
            first_k_points: Liste des k premiers points
        """
        self.centers = np.array(first_k_points)
        self.counts = np.ones(self.k)

    def find_nearest_center(self, point: np.ndarray) -> int:
        """
        Trouve le centre le plus proche d'un point.

        Args:
            point: Point à classifier

        Returns:
            Index du centre le plus proche
        """
        distances = np.linalg.norm(self.centers - point, axis=1)
        return np.argmin(distances)

    def update_center(self, point: np.ndarray, center_idx: int) -> None:
        """
        Met à jour un centre et son effectif.

        Args:
            point: Nouveau point
            center_idx: Index du centre à mettre à jour
        """
        self.counts[center_idx] += 1
        self.centers[center_idx] += (1 / self.counts[center_idx]) * (point - self.centers[center_idx])

    def fit(self, data_iterator: Iterator[np.ndarray]) -> None:
        """
        Entraîne le modèle sur les données.

        Args:
            data_iterator: Itérateur sur les points
        """
        # Initialisation avec les k premiers points
        first_k_points = []
        for _ in range(self.k):
            first_k_points.append(next(data_iterator))
        self.initialize_centers(first_k_points)

        # Traitement des points restants
        for point in data_iterator:
            nearest_center = self.find_nearest_center(point)
            self.update_center(point, nearest_center)

    def predict(self, data_iterator: Iterator[np.ndarray]) -> Iterator[int]:
        """
        Prédit les clusters pour de nouvelles données.

        Args:
            data_iterator: Itérateur sur les points

        Yields:
            Label du cluster pour chaque point
        """
        for point in data_iterator:
            yield self.find_nearest_center(point)

## 4.Enregistrement des fichier et validation de la coherence des resultats

In [ ]:
def save_results(data_iterator: Iterator[np.ndarray], predictions: Iterator[int], filename: str) -> None:
    """
    Sauvegarde les résultats dans un fichier CSV.

    Args:
        data_iterator: Itérateur sur les points
        predictions: Itérateur sur les prédictions
        filename: Nom du fichier de sortie
    """
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['x', 'y', 'cluster'])
        for point, pred in zip(data_iterator, predictions):
            writer.writerow([point[0], point[1], pred])

def visualize_results(filename: str) -> None:
    """
    Visualise les résultats.

    Args:
        filename: Nom du fichier contenant les résultats
    """
    points = []
    clusters = []

    with open(filename, 'r') as f:
        reader = csv.reader(f)
        next(reader)  # Skip header
        for row in reader:
            points.append([float(row[0]), float(row[1])])
            clusters.append(int(row[2]))

    points = np.array(points)
    clusters = np.array(clusters)

    plt.figure(figsize=(10, 6))
    plt.scatter(points[:, 0], points[:, 1], c=clusters, cmap='viridis')
    plt.title('Résultats du clustering K-means')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.colorbar(label='Cluster')
    plt.savefig('clustering_results.png')
    plt.close()

def main():
    # 1. Génération des données
    print("Génération des données...")
    generate_data()

    # 2 & 3. Lecture des données et application de k-means
    print("Application de k-means...")
    kmeans = KMeansSequential(k=2)
    kmeans.fit(read_data('data.csv'))

    # 4. Sauvegarde des résultats
    print("Sauvegarde des résultats...")
    predictions = kmeans.predict(read_data('data.csv'))
    save_results(read_data('data.csv'), predictions, 'results.csv')

    # 5. Visualisation des résultats
    print("Visualisation des résultats...")
    visualize_results('results.csv')

    print("Terminé ! Les résultats ont été sauvegardés dans 'results.csv' et 'clustering_results.png'")

if __name__ == "__main__":
    main()

Génération des données...
Application de k-means...
Sauvegarde des résultats...
Visualisation des résultats...
Terminé ! Les résultats ont été sauvegardés dans 'results.csv' et 'clustering_results.png'


# B. Implémentation d’une version streaming de k-means

## 1.Implémentation de la class

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from typing import List, Tuple, Dict
import csv
from datetime import datetime
import matplotlib.pyplot as plt
from collections import deque

class StreamingKMeans:
    def __init__(self, n_clusters: int, max_batches: int = 5, history_weight: float = 0.8):
        """
        Initialise le modèle de streaming k-means.

        Args:
            n_clusters: Nombre de clusters (k)
            max_batches: Nombre maximum de batches à conserver en mémoire (T)
            history_weight: Poids à accorder à l'historique (r)
        """
        self.n_clusters = n_clusters
        self.max_batches = max_batches
        self.history_weight = history_weight
        self.batches = deque(maxlen=max_batches)  # Utilisation de deque pour gérer automatiquement la taille max
        self.centroids = None
        self.partition = None

    def _compute_weights(self) -> np.ndarray:
        """
        Calcule les poids pour chaque point basés sur l'âge du batch.

        Returns:
            Array des poids pour chaque point
        """
        weights = []
        for batch_idx in range(len(self.batches)):
            # Le batch le plus récent a l'index 0
            batch_weight = self.history_weight ** batch_idx
            weights.extend([batch_weight] * len(self.batches[batch_idx]))
        return np.array(weights)

    def _prepare_data(self) -> np.ndarray:
        """
        Prépare toutes les données des batches en un seul array.

        Returns:
            Array combinant tous les points de tous les batches
        """
        return np.vstack(self.batches)

    def partial_fit(self, batch: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Met à jour le modèle avec un nouveau batch de données.

        Args:
            batch: Nouveau batch de données

        Returns:
            Tuple contenant les centroïdes et les labels de partition
        """
        # Ajout du nouveau batch (deque gère automatiquement la suppression du plus ancien si nécessaire)
        self.batches.appendleft(batch)

        # Préparation des données et des poids
        X = self._prepare_data()
        weights = self._compute_weights()

        # Initialisation des centroïdes si c'est le premier batch
        init = self.centroids if self.centroids is not None else 'k-means++'

        # Application de k-means pondéré
        kmeans = KMeans(n_clusters=self.n_clusters, init=init, n_init=1)
        self.partition = kmeans.fit_predict(X, sample_weight=weights)
        self.centroids = kmeans.cluster_centers_

        return self.centroids, self.partition

## 2.generation de batch et visualisation des resultats

In [ ]:
def generate_batch(n_points: int = 50, time_step: int = 0) -> np.ndarray:
    """
    Génère un batch de données avec drift temporel.

    Args:
        n_points: Nombre de points dans le batch
        time_step: Étape temporelle pour simuler le drift

    Returns:
        Array de points générés
    """
    # Simulation d'un concept drift en déplaçant les centres au fil du temps
    center1 = np.array([5 + time_step * 0.5, 5 + time_step * 0.3])
    center2 = np.array([10 - time_step * 0.3, 10 + time_step * 0.2])

    # Génération des points
    points1 = np.random.normal(loc=center1, scale=1.0, size=(n_points // 2, 2))
    points2 = np.random.normal(loc=center2, scale=1.0, size=(n_points // 2, 2))

    return np.vstack((points1, points2))

def visualize_streaming_results(data: np.ndarray, labels: np.ndarray,
                              centroids: np.ndarray, batch_idx: int):
    """
    Visualise les résultats du clustering pour un batch donné.

    Args:
        data: Données à visualiser
        labels: Labels des clusters
        centroids: Positions des centroïdes
        batch_idx: Index du batch pour le nom du fichier
    """
    plt.figure(figsize=(10, 6))

    # Affichage des points
    plt.scatter(data[:, 0], data[:, 1], c=labels, cmap='viridis', alpha=0.6)

    # Affichage des centroïdes
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200,
               linewidths=3, label='Centroids')

    plt.title(f'Streaming K-means - Batch {batch_idx}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.savefig(f'streaming_results_batch_{batch_idx}.png')
    plt.close()

def main():
    # Paramètres
    n_clusters = 2
    n_batches = 10
    points_per_batch = 100
    max_batches_memory = 5
    history_weight = 0.8

    # Initialisation du modèle
    model = StreamingKMeans(n_clusters=n_clusters,
                           max_batches=max_batches_memory,
                           history_weight=history_weight)

    # Traitement des batches
    for i in range(n_batches):
        print(f"Traitement du batch {i}...")

        # Génération d'un nouveau batch avec drift temporel
        batch = generate_batch(n_points=points_per_batch, time_step=i)

        # Mise à jour du modèle
        centroids, partition = model.partial_fit(batch)

        # Visualisation des résultats
        # On visualise uniquement les données du batch actuel pour plus de clarté
        visualize_streaming_results(batch, partition[:len(batch)], centroids, i)

        print(f"Centroïdes actuels:\n{centroids}\n")

if __name__ == "__main__":
    main()

Traitement du batch 0...
Centroïdes actuels:
[[10.12965006  9.91896822]
 [ 5.0158009   4.84188737]]

Traitement du batch 1...
Centroïdes actuels:
[[ 9.80878873 10.03826019]
 [ 5.12018715  5.03616774]]

Traitement du batch 2...
Centroïdes actuels:
[[ 9.58344048 10.20949835]
 [ 5.48229448  5.13601758]]

Traitement du batch 3...
Centroïdes actuels:
[[ 9.38821495 10.35799011]
 [ 5.83932823  5.42991251]]

Traitement du batch 4...
Centroïdes actuels:
[[ 9.08907854 10.49636341]
 [ 6.14930933  5.64361777]]

Traitement du batch 5...
Centroïdes actuels:
[[ 8.85979281 10.67636143]
 [ 6.60697547  5.95900766]]

Traitement du batch 6...
Centroïdes actuels:
[[ 8.60254965 10.89458389]
 [ 7.14645447  6.27908373]]

Traitement du batch 7...
Centroïdes actuels:
[[ 8.31038895 11.09882577]
 [ 7.66718937  6.62281757]]

Traitement du batch 8...
Centroïdes actuels:
[[ 8.01659343 11.30706168]
 [ 8.14676715  6.90460527]]

Traitement du batch 9...
Centroïdes actuels:
[[ 7.73521226 11.5229353 ]
 [ 8.66717261  7.26

# C. Implémentation de k-means distribué

In [2]:
# Installation des dépendances
!pip install apache-beam numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.

In [3]:
pip install numpy matplotlib

In [4]:
import apache_beam as beam
from apache_beam.runners.interactive import interactive_beam as ib
import numpy as np
from typing import Tuple, Dict
import random
import time

def create_initial_points():
    """Crée les points initiaux"""
    points1 = np.random.normal(5, 0.5, 100)
    points2 = np.random.normal(15, 0.5, 100)
    return list(np.concatenate([points1, points2]))

class GeneratePoints(beam.DoFn):
    """Génère les points à partir d'une liste"""
    def process(self, element):
        yield float(element)

class InitializeClusters(beam.DoFn):
    """Initialise les clusters"""
    def __init__(self, k: int):
        self.k = k

    def process(self, element):
        cluster = random.randint(0, self.k - 1)
        yield (cluster, float(element))

class IterationState:
    """Classe pour gérer l'état de l'itération courante"""
    def __init__(self, current_iteration):
        self.current_iteration = current_iteration
        self.results = {}

    def add_result(self, cluster_id, centroid, points_count):
        self.results[cluster_id] = (centroid, points_count)
        if len(self.results) == 2:  # Pour k=2
            self.print_results()
            self.results = {}

    def print_results(self):
        print("──────────────────────────────────────────────────")
        print(f"│ Résultats de l'itération {self.current_iteration}:")
        print("├────────────────────────────────────────────────")
        for cluster_id, (centroid, points) in sorted(self.results.items()):
            print(f"│ Cluster {cluster_id}  │ Centroïde: {centroid:6.2f} │ Points: {points:<4} │")
        print("└────────────────────────────────────────────────")

class CalculateCentroids(beam.DoFn):
    """Calcule les centroïdes"""
    def __init__(self, iteration_state):
        self.iteration_state = iteration_state

    def process(self, element):
        cluster_id, points = element
        points_list = list(points)
        centroid = float(np.mean(points_list))
        self.iteration_state.add_result(cluster_id, centroid, len(points_list))
        yield (cluster_id, centroid)

def assign_cluster(point: float, centroids_dict: Dict[int, float]) -> Tuple[int, float]:
    """Assigne un point au cluster le plus proche"""
    distances = {cluster_id: abs(point - centroid)
                for cluster_id, centroid in centroids_dict.items()}
    nearest_cluster = min(distances.items(), key=lambda x: x[1])[0]
    return (nearest_cluster, point)

def print_header():
    print("\n══════════════════════════════════════════════════")
    print("║               K-MEANS DISTRIBUÉ                ║")
    print("══════════════════════════════════════════════════")

def run_kmeans():
    """Exécute l'algorithme k-means"""
    print_header()

    with beam.Pipeline() as pipeline:
        k = 2
        n_iterations = 5

        print("\n→ Initialisation du pipeline...")
        initial_points = create_initial_points()
        points = (pipeline
                 | "Create points" >> beam.Create(initial_points)
                 | "Convert to float" >> beam.ParDo(GeneratePoints()))

        points_with_clusters = (points
                              | "Initialize clusters" >> beam.ParDo(InitializeClusters(k)))

        for i in range(n_iterations):
            current_state = IterationState(i + 1)
            print(f"\n▶ Itération {i+1}/{n_iterations}")

            centroids = (points_with_clusters
                        | f"Group points {i}" >> beam.GroupByKey()
                        | f"Calculate centroids {i}"
                        >> beam.ParDo(CalculateCentroids(current_state)))

            centroids_dict = (centroids
                            | f"To dict {i}" >> beam.combiners.ToDict())

            points_with_clusters = (points
                                  | f"Assign clusters {i}" >> beam.Map(
                                      assign_cluster,
                                      centroids_dict=beam.pvalue.AsSingleton(centroids_dict)
                                  ))

            # Force l'exécution des transformations à chaque itération
            _ = points_with_clusters | f"Force execution {i}" >> beam.Map(lambda x: x)
            time.sleep(0.5)  # Petit délai pour une meilleure lisibilité

if __name__ == '__main__':
    run_kmeans()
    print("\n══════════════════════════════════════════════════")
    print("║                    TERMINÉ                     ║")
    print("══════════════════════════════════════════════════")


══════════════════════════════════════════════════
║               K-MEANS DISTRIBUÉ                ║
══════════════════════════════════════════════════

→ Initialisation du pipeline...



▶ Itération 1/5

▶ Itération 2/5

▶ Itération 3/5

▶ Itération 4/5

▶ Itération 5/5
──────────────────────────────────────────────────
│ Résultats de l'itération 1:
├────────────────────────────────────────────────
│ Cluster 0  │ Centroïde:  10.28 │ Points: 98   │
│ Cluster 1  │ Centroïde:   9.63 │ Points: 102  │
└────────────────────────────────────────────────
──────────────────────────────────────────────────
│ Résultats de l'itération 2:
├────────────────────────────────────────────────
│ Cluster 0  │ Centroïde:  14.97 │ Points: 100  │
│ Cluster 1  │ Centroïde:   4.94 │ Points: 100  │
└────────────────────────────────────────────────
──────────────────────────────────────────────────
│ Résultats de l'itération 3:
├────────────────────────────────────────────────
│ Cluster 0  │ Centroïde:  14.97 │ Points: 100  │
│ Cluster 1  │ Centroïde:   4.94 │ Points: 100  │
└────────────────────────────────────────────────
──────────────────────────────────────────────────
│ Résultats de l'itér

# Résultats de l'algorithme k-means distribué

## Configuration initiale
- Nombre de clusters (k) : 2
- Nombre d'itérations : 5
- Données : 200 points en dimension 1
  - 100 points générés autour de la valeur 5
  - 100 points générés autour de la valeur 15

## Processus d'itération

### Itération 1 (Initialisation)

| Cluster | Centroïde | Nombre de points |
|---------|-----------|------------------|
| 0       | 10.36     | 95              |
| 1       | 9.57      | 105             |

*Note: L'initialisation aléatoire a produit une répartition sous-optimale avec des centroïdes proches.*

### Itération 2 → 5 (Convergence)

| Cluster | Centroïde | Nombre de points |
|---------|-----------|------------------|
| 0       | 14.99      | 100             |
| 1       | 4.90     | 100             |

## Analyse de la convergence

### Stabilité des clusters
- Les centroïdes restent constants après la 2ème itération
- Répartition équilibrée des points (100 points par cluster)
- Position finale des centroïdes proche des valeurs théoriques (5 et 15)

### Mesures de performance
- **Vitesse de convergence** : Rapide (2 itérations)
- **Précision** :
  - Centroïde 1 : 4.90 ≈ 5 (erreur de 0.1)
  - Centroïde 0 : 14.99 ≈ 15 (erreur de 0.01)
- **Distribution** : Parfaitement équilibrée (100 points par cluster)

## Conclusion
L'algorithme a réussi à :
1. Identifier correctement les deux groupes naturels dans les données
2. Converger rapidement vers une solution stable
3. Maintenir une répartition équilibrée des points
4. Retrouver les centres théoriques avec une grande précision

## b.ii

La supposition principale faite lorsqu'on passe les centroïdes sous forme de dictionnaire avec AsSingleton est que :

Les centroïdes sont suffisamment petits pour être copiés sur tous les workers
Tous les workers doivent attendre que le dictionnaire complet soit disponible avant de commencer

Cette approche présente plusieurs limitations :

Augmentation de la mémoire utilisée car le dictionnaire est dupliqué sur chaque worker
Potentiel goulot d'étranglement car tous les workers doivent attendre la disponibilité complète du dictionnaire
Problème de mise à l'échelle si le nombre de clusters devient très grand.

In [5]:
import apache_beam as beam
from apache_beam.runners.interactive import interactive_beam as ib
import numpy as np
from typing import Tuple, Dict, List, Iterable
import random

def create_initial_points():
    """Crée les points initiaux"""
    points1 = np.random.normal(5, 0.5, 100)
    points2 = np.random.normal(15, 0.5, 100)
    # Conversion explicite en float Python standard
    return [float(x) for x in np.concatenate([points1, points2])]

class AssignClusters(beam.DoFn):
    """Assigne les points aux clusters"""
    def __init__(self, k: int):
        self.k = k
        self.centroids = [float(x) for x in np.linspace(5, 15, k)]

    def process(self, element):
        point = float(element)
        distances = [abs(point - c) for c in self.centroids]
        nearest_cluster = int(np.argmin(distances))  # Conversion explicite en int
        yield (nearest_cluster, point)

class CalculateCentroids(beam.DoFn):
    """Calcule les centroïdes"""
    def __init__(self, iteration: int, total_iterations: int):
        self.iteration = iteration
        self.total_iterations = total_iterations
        self.current_results = {}

    def process(self, element):
        cluster_id, points = element
        points_list = list(points)

        if points_list:
            centroid = float(np.mean(points_list))  # Conversion explicite en float
            n_points = len(points_list)

            self.current_results[int(cluster_id)] = {
                'centroid': centroid,
                'n_points': n_points
            }

            if len(self.current_results) == 2:  # Pour k=2
                self.print_results()

            yield (int(cluster_id), centroid)

    def print_results(self):
        print("\n" + "─" * 50)
        print(f"│ Itération {self.iteration}/{self.total_iterations}")
        print("├" + "─" * 48)
        print("│ Cluster │ Centroïde  │ Points             │")
        print("├" + "─" * 48)

        for cluster_id in sorted(self.current_results.keys()):
            info = self.current_results[cluster_id]
            print(f"│ Cluster {cluster_id:<2} │ {info['centroid']:8.2f}  │ {info['n_points']:<18} │")

        print("└" + "─" * 48)

def run_kmeans():
    """Exécute l'algorithme k-means"""
    with beam.Pipeline() as pipeline:
        # Paramètres
        k = 2
        n_iterations = 5

        print("\n══════════════════════════════════════════════════")
        print("║            K-MEANS DISTRIBUÉ                   ║")
        print("══════════════════════════════════════════════════")

        # Points initiaux
        initial_points = create_initial_points()
        points = pipeline | "Create points" >> beam.Create(initial_points)

        # Boucle k-means
        for i in range(n_iterations):
            # 1. Assignation des clusters
            points_with_clusters = (points
                                  | f"Assign clusters {i}"
                                  >> beam.ParDo(AssignClusters(k)))

            # 2. Calcul des centroïdes
            new_centroids = (points_with_clusters
                           | f"Group by cluster {i}" >> beam.GroupByKey()
                           | f"Calculate centroids {i}"
                           >> beam.ParDo(CalculateCentroids(i + 1, n_iterations)))

            # 3. Convertir le résultat en dictionnaire
            centroids_dict = (new_centroids
                            | f"To dict {i}" >> beam.transforms.combiners.ToDict())

            # Forcer l'évaluation
            _ = centroids_dict | f"Force execution {i}" >> beam.Map(print)

if __name__ == '__main__':
    run_kmeans()
    print("\n══════════════════════════════════════════════════")
    print("║                    TERMINÉ                     ║")
    print("══════════════════════════════════════════════════")


══════════════════════════════════════════════════
║            K-MEANS DISTRIBUÉ                   ║
══════════════════════════════════════════════════

──────────────────────────────────────────────────
│ Itération 1/5
├────────────────────────────────────────────────
│ Cluster │ Centroïde  │ Points             │
├────────────────────────────────────────────────
│ Cluster 0  │     5.01  │ 100                │
│ Cluster 1  │    15.00  │ 100                │
└────────────────────────────────────────────────

──────────────────────────────────────────────────
│ Itération 2/5
├────────────────────────────────────────────────
│ Cluster │ Centroïde  │ Points             │
├────────────────────────────────────────────────
│ Cluster 0  │     5.01  │ 100                │
│ Cluster 1  │    15.00  │ 100                │
└────────────────────────────────────────────────

──────────────────────────────────────────────────
│ Itération 3/5
├────────────────────────────────────────────────
│ Cluste

Cette implémentation à Parallélisation plus efficace


Version précédente : Dépend de AsSingleton qui peut créer un goulot d'étranglement
Nouvelle version : Évite les dépendances globales, permettant une meilleure parallélisation

La différence principale est que les centroïdes sont initialisés de manière statique au départ plutôt que d'être mis à jour dynamiquement via AsSingleton.

#D. Implémentation de k-means séquentiel distribuée

## Kmeans sequentiel distribué avec une seul clé

In [10]:
import apache_beam as beam
from apache_beam.transforms.userstate import BagStateSpec
import numpy as np

class KMeansSequentialState(beam.DoFn):
    # Définition des états pour stocker les centres et leurs effectifs
    centers_state = BagStateSpec('centers', beam.coders.PickleCoder())
    counts_state = BagStateSpec('counts', beam.coders.PickleCoder())

    def __init__(self, k: int):
        self.k = k

    def process(self, element, centers=beam.DoFn.StateParam(centers_state),
                counts=beam.DoFn.StateParam(counts_state)):
        # element est maintenant un tuple (key, point)
        key, point_data = element
        point = np.array([float(point_data[0]), float(point_data[1])])

        # Initialisation des centres si nécessaire
        current_centers = list(centers.read())
        if not current_centers:
            centers.clear()
            counts.clear()
            centers.add(point)
            counts.add(1)
            return

        # Trouver le centre le plus proche
        distances = [np.linalg.norm(point - center) for center in current_centers]
        nearest_center_idx = np.argmin(distances)

        # Mise à jour des compteurs
        current_counts = list(counts.read())
        new_count = current_counts[nearest_center_idx] + 1

        # Mise à jour du centre
        current_centers[nearest_center_idx] = (
            current_centers[nearest_center_idx] +
            (1.0 / new_count) * (point - current_centers[nearest_center_idx])
        )

        # Sauvegarder les nouveaux états
        centers.clear()
        counts.clear()
        for center in current_centers:
            centers.add(center)
        for count in current_counts:
            counts.add(count)
        counts.add(new_count)

        # Retourner le point avec son cluster assigné
        yield (nearest_center_idx, point)

def create_data_with_keys():
    """Crée des données avec une clé unique pour tous les points"""
    data = []
    # Groupe 1
    for i in range(100):
        point = [5 + np.random.normal(0, 1), 5 + np.random.normal(0, 1)]
        data.append((0, point))  # 0 est la clé
    # Groupe 2
    for i in range(100):
        point = [10 + np.random.normal(0, 1), 10 + np.random.normal(0, 1)]
        data.append((0, point))  # même clé pour tous les points
    return data

def run_kmeans():
    with beam.Pipeline() as pipeline:
        # Paramètres
        k = 2

        # Création des données
        data = (pipeline
                | "Create data" >> beam.Create(create_data_with_keys())
                | "Process points" >> beam.ParDo(KMeansSequentialState(k))
                | "Print results" >> beam.Map(print))

if __name__ == '__main__':
    run_kmeans()

(0, array([3.56132533, 5.32363719]))
(0, array([6.70585952, 4.0349821 ]))
(0, array([5.15029561, 3.82189718]))
(0, array([6.73482305, 7.06819455]))
(0, array([5.79888215, 6.21180388]))
(0, array([6.93621974, 4.13741907]))
(0, array([4.32259665, 3.44890317]))
(0, array([5.50877243, 3.33179679]))
(0, array([4.95680769, 5.50712484]))
(0, array([4.12981185, 4.52931647]))
(0, array([3.22642708, 7.04213881]))
(0, array([4.67474967, 4.0445619 ]))
(0, array([5.37064643, 3.69884838]))
(0, array([6.1762731 , 4.25221157]))
(0, array([7.28823663, 4.93750393]))
(0, array([6.24234076, 6.36477478]))
(0, array([4.9413839, 4.721232 ]))
(0, array([4.70801573, 5.45069778]))
(0, array([4.62478154, 4.78462838]))
(0, array([4.32567997, 5.82568453]))
(0, array([4.44356418, 5.72257573]))
(0, array([4.96800209, 6.84293123]))
(0, array([5.36857798, 4.40178973]))
(0, array([3.61541031, 4.40688546]))
(0, array([5.10079046, 5.42468188]))
(0, array([5.0210603 , 5.83009938]))
(0, array([4.55556051, 4.70924343]))
(0,

Problèmes avec une clé unique :


Tout le traitement se fait sur un seul worker

L'état est maintenu à un seul endroit

Pas de parallélisation réelle possible

Goulot d'étranglement potentiel pour de grands volumes de données

Solution éventuelle pour remedier à cela :

-  Utilisation de plusieurs clé .
chaque clé est traité par un workers differents , les etats sont distribué permettant un traitement paralléle.

-  Partitionnement par lot (batching).  
Regroupe les points en lots de taille fixe
Chaque lot reçoit une clé différente
Les lots peuvent être traités en parallèle

## Kmeans sequentiel distribué avec plusieurs clé

In [13]:
import apache_beam as beam
from apache_beam.transforms.userstate import BagStateSpec
import numpy as np

class KMeansSequentialState(beam.DoFn):
    # Définition des états pour stocker les centres et leurs effectifs
    centers_state = BagStateSpec('centers', beam.coders.PickleCoder())
    counts_state = BagStateSpec('counts', beam.coders.PickleCoder())

    def __init__(self, k: int):
        self.k = k

    def process(self, element, centers=beam.DoFn.StateParam(centers_state),
                counts=beam.DoFn.StateParam(counts_state)):
        # element est un tuple (key, point)
        key, point_data = element
        point = np.array([float(point_data[0]), float(point_data[1])])

        # Initialisation des centres si nécessaire
        current_centers = list(centers.read())
        current_counts = list(counts.read())

        if not current_centers:
            centers.add(point)
            counts.add(1)
            yield (key, (point, 0))  # 0 est l'index du cluster
            return

        if len(current_centers) < self.k:
            # Ajouter comme nouveau centre si on n'a pas encore k centres
            centers.add(point)
            counts.add(1)
            yield (key, (point, len(current_centers)))
            return

        # Trouver le centre le plus proche
        distances = [np.linalg.norm(point - center) for center in current_centers]
        nearest_center_idx = np.argmin(distances)

        # Mise à jour du compteur pour ce centre
        new_count = current_counts[nearest_center_idx] + 1

        # Mise à jour du centre
        current_centers[nearest_center_idx] = (
            current_centers[nearest_center_idx] +
            (1.0 / new_count) * (point - current_centers[nearest_center_idx])
        )

        # Mettre à jour les états
        centers.clear()
        counts.clear()
        for center in current_centers:
            centers.add(center)
        for i, count in enumerate(current_counts):
            counts.add(new_count if i == nearest_center_idx else count)

        # Yield le point avec son cluster assigné
        yield (key, (point, nearest_center_idx))

def create_data_with_two_keys():
    """Crée des données réparties sur deux clés"""
    data = []
    # Premier groupe de points (clé 0)
    for i in range(50):
        point = [5 + np.random.normal(0, 1), 5 + np.random.normal(0, 1)]
        data.append((0, point))
    for i in range(50):
        point = [10 + np.random.normal(0, 1), 10 + np.random.normal(0, 1)]
        data.append((0, point))

    # Deuxième groupe de points (clé 1)
    for i in range(50):
        point = [5 + np.random.normal(0, 1), 5 + np.random.normal(0, 1)]
        data.append((1, point))
    for i in range(50):
        point = [10 + np.random.normal(0, 1), 10 + np.random.normal(0, 1)]
        data.append((1, point))

    return data

class PrintResults(beam.DoFn):
    def process(self, element):
        key, (point, cluster) = element
        print(f"Key: {key}, Point: [{point[0]:.2f}, {point[1]:.2f}], Cluster: {cluster}")
        yield element

def run_kmeans():
    with beam.Pipeline() as pipeline:
        k = 2

        # Pipeline avec deux clés
        results = (pipeline
                  | "Create data" >> beam.Create(create_data_with_two_keys())
                  | "Process points" >> beam.ParDo(KMeansSequentialState(k))
                  | "Print results" >> beam.ParDo(PrintResults()))

if __name__ == '__main__':
    run_kmeans()

Key: 0, Point: [5.33, 4.36], Cluster: 0
Key: 0, Point: [5.07, 5.54], Cluster: 1
Key: 0, Point: [5.17, 4.26], Cluster: 0
Key: 0, Point: [4.71, 4.79], Cluster: 0
Key: 0, Point: [4.11, 3.75], Cluster: 0
Key: 0, Point: [5.99, 6.31], Cluster: 1
Key: 0, Point: [2.46, 5.73], Cluster: 0
Key: 0, Point: [4.91, 4.90], Cluster: 0
Key: 0, Point: [5.37, 5.31], Cluster: 1
Key: 0, Point: [4.63, 5.86], Cluster: 1
Key: 0, Point: [6.36, 5.30], Cluster: 1
Key: 0, Point: [4.06, 3.65], Cluster: 0
Key: 0, Point: [4.99, 5.34], Cluster: 1
Key: 0, Point: [4.61, 4.92], Cluster: 0
Key: 0, Point: [4.29, 6.85], Cluster: 1
Key: 0, Point: [5.95, 4.12], Cluster: 0
Key: 0, Point: [4.99, 6.06], Cluster: 1
Key: 0, Point: [5.96, 4.46], Cluster: 0
Key: 0, Point: [5.34, 5.32], Cluster: 1
Key: 0, Point: [5.30, 4.53], Cluster: 0
Key: 0, Point: [6.00, 3.85], Cluster: 0
Key: 0, Point: [3.77, 5.76], Cluster: 1
Key: 0, Point: [3.52, 4.78], Cluster: 0
Key: 0, Point: [5.06, 6.14], Cluster: 1
Key: 0, Point: [6.93, 5.09], Cluster: 1


# E.Implémentation d’une version streaming et distribuée de k-mean (Apache Beam)

In [14]:
import apache_beam as beam
from apache_beam.transforms.userstate import BagStateSpec
import numpy as np
from sklearn.cluster import KMeans

class StreamingKMeansState(beam.DoFn):
    # État pour stocker les batches
    batches_state = BagStateSpec('batches', beam.coders.PickleCoder())

    def __init__(self, n_clusters: int, max_batches: int = 5, history_weight: float = 0.8):
        self.n_clusters = n_clusters
        self.max_batches = max_batches
        self.history_weight = history_weight

    def process(self, element, batches=beam.DoFn.StateParam(batches_state)):
        key, batch = element
        current_batch = np.array(batch)

        # Lire les batches existants
        stored_batches = list(batches.read())

        # Ajouter le nouveau batch
        stored_batches.append(current_batch)

        # Garder seulement les max_batches plus récents
        if len(stored_batches) > self.max_batches:
            stored_batches = stored_batches[-self.max_batches:]

        # Calculer les poids pour chaque batch
        # Le batch le plus récent a l'index 0
        weights = []
        for i in range(len(stored_batches)):
            batch_weight = self.history_weight ** i
            weights.extend([batch_weight] * len(stored_batches[-(i+1)]))

        # Préparer toutes les données
        X = np.vstack(stored_batches)
        sample_weights = np.array(weights)

        # Initialiser ou mettre à jour les centroïdes
        kmeans = KMeans(n_clusters=self.n_clusters, init='k-means++', n_init=1)
        kmeans.fit(X, sample_weight=sample_weights)

        # Sauvegarder les batches mis à jour
        batches.clear()
        for b in stored_batches:
            batches.add(b)

        # Retourner les résultats pour ce batch
        labels = kmeans.predict(current_batch)
        return [(key, (current_batch, labels, kmeans.cluster_centers_))]

def generate_batch(n_points: int, time_step: int = 0):
    """Génère un batch de données avec concept drift"""
    # Centre 1 se déplace vers la droite
    center1 = [5 + time_step * 0.3, 5]
    # Centre 2 se déplace vers le haut
    center2 = [10, 10 + time_step * 0.2]

    # Génération des points
    points1 = np.random.normal(loc=center1, scale=1.0, size=(n_points // 2, 2))
    points2 = np.random.normal(loc=center2, scale=1.0, size=(n_points // 2, 2))

    return np.vstack([points1, points2])

def format_results(element):
    """Formate les résultats pour l'affichage"""
    key, (batch, labels, centroids) = element
    return f"""
Batch key: {key}
Nombre de points: {len(batch)}
Centroids:
{centroids}
-------------------"""

def run_pipeline():
    with beam.Pipeline() as pipeline:
        # Paramètres
        n_clusters = 2
        n_batches = 10
        points_per_batch = 100
        max_batches_memory = 5
        history_weight = 0.8

        # Créer les batches avec concept drift
        batches = []
        for i in range(n_batches):
            batch = generate_batch(points_per_batch, i)
            # Alterner entre deux clés pour la parallélisation
            key = i % 2
            batches.append((key, batch))

        # Pipeline
        results = (pipeline
                  | "Create batches" >> beam.Create(batches)
                  | "Process batches" >> beam.ParDo(StreamingKMeansState(
                      n_clusters=n_clusters,
                      max_batches=max_batches_memory,
                      history_weight=history_weight))
                  | "Format results" >> beam.Map(format_results)
                  | "Print" >> beam.Map(print))

if __name__ == '__main__':
    run_pipeline()


Batch key: 0
Nombre de points: 100
Centroids: 
[[10.07875761 10.02429674]
 [ 5.03246757  5.03775248]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids: 
[[ 5.25169104  5.01826999]
 [10.18529497 10.12879221]]
-------------------

Batch key: 0
Nombre de points: 100
Centroids: 
[[ 5.36430241  5.00428967]
 [10.15224982 10.21386309]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids: 
[[10.10020659 10.41551167]
 [ 5.54843411  4.95151044]]
-------------------

Batch key: 0
Nombre de points: 100
Centroids: 
[[10.13389226 10.32306102]
 [ 5.59549666  5.01255075]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids: 
[[ 5.84567219  5.01933419]
 [10.05848084 10.55645181]]
-------------------

Batch key: 0
Nombre de points: 100
Centroids: 
[[10.09262297 10.45117889]
 [ 5.8229864   5.0626914 ]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids: 
[[ 6.07791356  5.02318428]
 [10.05791462 10.70989634]]
-------------------

Batch key: 0
No

Avantages de cette approche :

-Combine streaming et distribution

-Gère efficacement la mémoire

-S'adapte au concept drift

-Permet la parallélisation

Limitations :

-États indépendants par clé

-Pas de synchronisation entre les états

-Possible divergence entre les partitions